# LaTeX tables

Results, backbones and ablations, written to `tables/*.tex`.

In [ ]:
import json
from decimal import ROUND_HALF_DOWN, ROUND_HALF_UP, Decimal
from pathlib import Path

import utils as u

ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/tables"

In [ ]:
data = json.loads((data_path / "data.json").read_text())
figs_path.mkdir(exist_ok=True)

In [ ]:
def tex_cell(v: float, decimals: int = 0, sign: bool = False, bold: bool = False, colour: str | None = None) -> str:
    # ties round towards +inf on the decimal value (format() rounds the binary float: 90.55 gave 90.5),
    # so a difference to an exact value equals the difference of the displayed values
    d = Decimal(repr(round(v, 9)))
    q = d.quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP if d >= 0 else ROUND_HALF_DOWN)
    txt = (f"{q:+}" if sign else f"{q}").replace("-", "−")
    if bold:
        txt = rf"\textbf{{{txt}}}"
    return rf"\textcolor{{{colour}}}{{{txt}}}" if colour else txt


def tex_label(label: str) -> str:
    return (label.replace("π0.5", r"$\pi_{0.5}$").replace("π0-FAST", r"$\pi_0$-FAST").replace("π0", r"$\pi_0$")
            .replace("k-NN", r"\knnword{}").replace("kNN", r"\knnword{}"))  # fmt: skip


CITES = {
    "π0.5": "pi2025pi05",
    "MolmoAct2": "fang2026molmoact2",
    "GR00T N1.7": "nvidia2025gr00t",
    "VLA-JEPA": "sun2026vlajepa",
    "Evo-1": "lin2026evo1",
    "X-VLA": "zheng2026xvla",
    "π0": "black2024pi0",
    "SmolVLA": "shukor2025smolvla",
    "π0-FAST": "pertsch2025fast",
}

## Results

LIBERO and LIBERO-plus success of every policy, average and per suite.

In [ ]:
rows = [(lab, role, u.libero_success(data, run), u.plus_success(data, spec)) for lab, role, run, spec in u.ranked_policies(data)]
cols = [[r[2][j] for r in rows] for j in range(5)] + [[r[3][j] for r in rows] for j in range(5)]
best = [max(c) for c in cols]
min_drop = min(r[2][0] - r[3][0] for r in rows)
knn_order = ["k-NN (ResNet)", "k-NN (multimodal)", "k-NN (LAC key)"]
vlas = [i for i, r in enumerate(rows) if r[1] == "vla"]
knns = sorted(
    (i for i, r in enumerate(rows) if r[1] != "vla"),
    key=lambda i: knn_order.index(rows[i][0]),
)
lines = [r"\begin{table}[t!]",
         r"\centering",
         r"\caption{Success rate (\unit{\percent}) on LIBERO (official protocol, \num{2000} episodes) and on the same 420 LIBERO-Plus variants. \acp{VLA} are ordered by average on LIBERO-Plus. $\Delta$ is the difference between the average success rates on LIBERO-Plus and LIBERO, computed before rounding. The \ac{kNN} rows are training-free, except for the \ac{LAC} key.}\label{tab:results}",
         r"\scriptsize",
         r"\setlength{\tabcolsep}{5pt}",
         r"\resizebox{\linewidth}{!}{%",
         r"  \begin{tabular}{@{}l*{5}{r}@{\hspace{1.5mm}}*{5}{r}@{\hspace{1.5mm}}r@{}}",
         r"  \toprule",
         r"  & \multicolumn{5}{c}{LIBERO}",
         r"  & \multicolumn{5}{c}{LIBERO-Plus}",
         r"  & \\",
         r"  \cmidrule(lr){2-6}\cmidrule(lr){7-11}",
         r"  & avg & Spat & Obj & Goal & Long",
         r"  & avg & Spat & Obj & Goal & Long & $\Delta$ \\",
         r"  \midrule",
         ""]  # fmt: skip
for group in (vlas, knns):
    for i in group:
        lab, role, clean, plus = rows[i]
        cells = [
            tex_cell(
                cols[j][i],
                1,
                bold=cols[j][i] >= best[j] - 1e-9,
            )
            for j in range(10)
        ]
        drop = clean[0] - plus[0]
        delta = tex_cell(
            -drop,
            1,
            sign=True,
            bold=drop <= min_drop + 1e-9,
        ).replace("−", "-")
        if role == "vla":
            label = tex_label(lab) + rf"~\citep{{{CITES[lab]}}}"
        else:
            name = lab.removeprefix("k-NN (").removesuffix(")")
            label = rf"\rowcolor{{tint}}\knnword{{}} (\emph{{{name}}})"
        lines += ["  " + label,
                  "  & " + " & ".join(cells[:5]),
                  "  & " + " & ".join(cells[5:]) + f" & {delta} \\\\",
                  ""]  # fmt: skip
    if group is vlas:
        lines += [r"  \midrule", ""]
lines += [r"  \bottomrule",
          r"  \end{tabular}%",
          "}",
          r"\end{table}"]  # fmt: skip
(figs_path / "results_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Backbones

The lookup keyed on each VLA's own VLM before / after LIBERO fine-tuning, next to the VLA. Orange: the lookup on the fine-tuned VLM beats the VLA itself.

In [ ]:
lines = [r"\begin{tabularx}{\linewidth}{@{}l*{6}{>{\raggedleft\arraybackslash}X}@{}}", r"\toprule",
         r" & \multicolumn{3}{c}{LIBERO} & \multicolumn{3}{c}{LIBERO-plus} \\",
         r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
         r" & \knnword{} before & \knnword{} after & VLA & \knnword{} before & \knnword{} after & VLA \\", r"\midrule"]  # fmt: skip
for label, before, after, vla in u.backbone_rows(data):
    cells = []
    for col in (0, 1):
        b, a, v = before[col], after[col], vla[col]
        beats = a >= v
        cells += [tex_cell(b, 1), tex_cell(a, 1, bold=beats, colour="knn" if beats else None),
                  tex_cell(v, 1, colour="learned")]  # fmt: skip
    lines.append(tex_label(label) + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabularx}"]
(figs_path / "backbones_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Ablations

One change to the full method at a time, on the official protocol (2,000 episodes).

In [ ]:
ABLATIONS = {  # config stem in outputs/knn/ablations -> label, in the table's order
    "no_language": "prompt ignored", "encoder_pixels": "raw 16×16 pixels", "demos_10": "10% of demos",
    "demos_25": "25% of demos", "demos_50": "50% of demos", "k1": "k = 1 (replay nearest demo)",
    "encoder_dinov2": "DINOv2-S instead of ResNet-18", "no_wrist": "agent-view camera only",
    "no_agentview": "wrist camera only", "no_mode": "plain weighted mean", "no_ensemble": "no temporal ensembling",
    "no_history": "no history matching", "no_state": "no proprioceptive state", "no_time": "no time-step feature",
    "hard_filter": "hard task filter",
}  # fmt: skip

In [ ]:
abf, full = data["ablations_full"], u.libero_success(data, "knn:00_base")
labels = [(label.replace("%", r"\%").replace("×", r"$\times$"), stem) for stem, label in ABLATIONS.items()]
labels_k = [(f"$k$ = {int(k[1:])} (13 in the full method)", k) for k in ("k02", "k04", "k08", "k24", "k32")]
lines = [r"\begin{tabularx}{\linewidth}{@{}X*{6}{r}@{}}", r"\toprule",
         r"Change to the full method & avg & Spatial & Object & Goal & Long & $\Delta$ \\", r"\midrule",
         r"none (full method, \texttt{lerobot-eval}) & " + " & ".join(tex_cell(v, 1) for v in full) + r" & \\"]  # fmt: skip
for group in (labels, labels_k):
    for label, stem in group:
        r = abf[stem]
        cells = [tex_cell(v, 1) for v in (r["success"], *r["suites"])]
        lines.append(f"{label} & " + " & ".join(cells) + f" & {tex_cell(r['success'] - full[0], 1, sign=True)}" + r" \\")
    if group is labels:
        lines.append(r"\midrule")
lines += [r"\bottomrule", r"\end{tabularx}"]
(figs_path / "ablations_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))